# VQE for H2 — the honest version of Q-VIRA's drug-discovery track

**Why this notebook exists.**

The original Q-VIRA dashboard displayed a "Ground State Energy (E0)" of
`-(4.2 + threat_score)` Hartree, computed in JavaScript, labelled as a
12-qubit Qiskit VQE of a viral binding pocket with a Paxlovid derivative.
No VQE was run and no molecule was simulated.

This notebook replaces that claim with something real and small enough to
verify: a genuine VQE on H2 and LiH, **benchmarked against exact
diagonalisation** at every bond length.

**What a judge should take away:** we know the difference between what VQE
can do today (a few qubits, small active spaces) and what the marketing
version claimed (protein-ligand binding). Stating that boundary clearly is
worth more than an impressive number nobody can reproduce.


In [ ]:
import pennylane as qml
from pennylane import numpy as np
import numpy as onp
import matplotlib.pyplot as plt

print("PennyLane", qml.version())

## 1. Build the molecular Hamiltonian

`qml.qchem.molecular_hamiltonian` runs a PySCF Hartree-Fock calculation and
maps the second-quantised fermionic Hamiltonian onto qubits (Jordan-Wigner by
default). For H2 in STO-3G this gives **4 spin-orbitals = 4 qubits**.

This is the same qubit count as our threat-scoring VQC, which is a useful
coincidence for the pitch: it shows the scale we are actually working at.

In [ ]:
def h2_hamiltonian(bond_length):
    symbols = ["H", "H"]
    coords = np.array([[0.0, 0.0, -bond_length / 2],
                       [0.0, 0.0,  bond_length / 2]])
    H, n_qubits = qml.qchem.molecular_hamiltonian(
        symbols, coords, basis="sto-3g", unit="angstrom"
    )
    return H, n_qubits

H, n_qubits = h2_hamiltonian(0.7414)   # experimental equilibrium bond length
print("qubits:", n_qubits)
print(H)

## 2. The reference: exact diagonalisation

For 4 qubits the Hamiltonian is a 16x16 matrix, so we can diagonalise it
exactly. **This is the number VQE must reproduce.** Without this column, a
VQE energy is unfalsifiable -- which is exactly the problem with the original
dashboard.

In [ ]:
def exact_ground_state(H):
    return float(onp.linalg.eigvalsh(qml.matrix(H))[0])

print(f"exact ground state: {exact_ground_state(H):.8f} Ha")

## 3. The VQE

We use the UCCSD-style `AllSinglesDoubles` ansatz starting from the
Hartree-Fock state. Parameters are optimised by gradient descent on the
energy expectation value -- the variational principle guarantees the result
is an upper bound on the true ground state.

In [ ]:
def run_vqe(H, n_qubits, n_electrons=2, steps=60, stepsize=0.4, verbose=False):
    dev = qml.device("default.qubit", wires=n_qubits)
    hf_state = qml.qchem.hf_state(n_electrons, n_qubits)
    singles, doubles = qml.qchem.excitations(n_electrons, n_qubits)

    @qml.qnode(dev)
    def circuit(params):
        qml.AllSinglesDoubles(params, wires=range(n_qubits),
                              hf_state=hf_state, singles=singles, doubles=doubles)
        return qml.expval(H)

    params = np.zeros(len(singles) + len(doubles), requires_grad=True)
    opt = qml.GradientDescentOptimizer(stepsize)
    history = []
    for i in range(steps):
        params, energy = opt.step_and_cost(circuit, params)
        history.append(float(energy))
        if verbose and i % 10 == 0:
            print(f"  step {i:3d}  E = {energy:.8f} Ha")
    return float(circuit(params)), params, history

energy, params, history = run_vqe(H, n_qubits, verbose=True)
exact = exact_ground_state(H)
print(f"\nVQE   : {energy:.8f} Ha")
print(f"exact : {exact:.8f} Ha")
print(f"error : {abs(energy - exact):.2e} Ha  (chemical accuracy = 1.6e-3)")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history, label="VQE energy")
plt.axhline(exact, color="crimson", ls="--", label="exact diagonalisation")
plt.xlabel("optimiser step"); plt.ylabel("energy (Ha)")
plt.title("VQE convergence, H$_2$ at 0.7414 A")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. The dissociation curve

A single energy at one geometry proves little. Scanning the bond length is
the standard test: it checks the ansatz across the whole potential energy
surface, including the stretched geometries where correlation is strongest
and simple methods break down.

**Expected result:** error stays below chemical accuracy (1.6 mHa) at every
point, and the curve has its minimum near the experimental 0.74 A.

In [ ]:
bond_lengths = [0.4, 0.5, 0.6, 0.7414, 0.9, 1.1, 1.3, 1.5, 1.8, 2.1]
vqe_energies, exact_energies = [], []

for R in bond_lengths:
    Hr, nq = h2_hamiltonian(R)
    e_vqe, _, _ = run_vqe(Hr, nq)
    e_exact = exact_ground_state(Hr)
    vqe_energies.append(e_vqe); exact_energies.append(e_exact)
    print(f"R={R:5.3f} A   VQE={e_vqe:.8f}   exact={e_exact:.8f}   err={abs(e_vqe-e_exact):.2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(bond_lengths, exact_energies, "o-", label="exact")
ax1.plot(bond_lengths, vqe_energies, "x--", label="VQE")
ax1.set_xlabel("bond length (A)"); ax1.set_ylabel("energy (Ha)")
ax1.set_title("H$_2$ dissociation curve"); ax1.legend(); ax1.grid(alpha=.3)

errors = [abs(v - e) for v, e in zip(vqe_energies, exact_energies)]
ax2.semilogy(bond_lengths, errors, "s-", color="darkorange")
ax2.axhline(1.6e-3, color="crimson", ls="--", label="chemical accuracy")
ax2.set_xlabel("bond length (A)"); ax2.set_ylabel("|VQE - exact| (Ha)")
ax2.set_title("VQE error"); ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()
print(f"max error across the curve: {max(errors):.2e} Ha")

## 5. Why this does NOT scale to a drug-binding pocket

This is the slide that earns credibility rather than losing it.

| | H2 (this notebook) | SARS-CoV-2 Mpro + inhibitor |
|---|---|---|
| atoms | 2 | ~5,000 (protein) + ~30 (ligand) |
| qubits (STO-3G, all-electron) | 4 | ~10^5 |
| exact reference available? | yes, 16x16 matrix | no |
| runs on a laptop | yes | no, and not on any existing hardware |

Qubit count in a minimal basis grows as roughly 2x the number of spatial
orbitals, so a binding pocket is many orders of magnitude beyond today's
devices -- before you even add error correction. Any demo claiming a VQE
computed a protein-ligand binding energy is doing something else.

**The credible roadmap**, in order:

1. H2 / LiH benchmarked against exact diagonalisation *(this notebook)*
2. Small active spaces (CAS(2,2), CAS(4,4)) of a real ligand fragment,
   benchmarked against CASCI
3. Embedding a small quantum active space inside a classical DFT/MM
   calculation of the pocket
4. Protein-ligand binding free energies — **not a near-term quantum target**

Say step 4 is out of reach. A jury that works in this field already knows,
and saying it first is what separates a serious team from a demo.

## 6. Optional: LiH (12 qubits)

LiH in STO-3G needs 12 qubits -- the number the old dashboard displayed. Run
this to see what 12 qubits actually buys you: a 3-atom molecule, not a
binding pocket.

Exact diagonalisation of a 4096x4096 matrix is still fine on a laptop, so you
keep the benchmark column.

In [ ]:
# Takes a few minutes. Uncomment to run.
# symbols = ["Li", "H"]
# coords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.595]])
# H_lih, nq_lih = qml.qchem.molecular_hamiltonian(symbols, coords, basis="sto-3g", unit="angstrom")
# print("LiH qubits:", nq_lih)
# e_vqe_lih, _, _ = run_vqe(H_lih, nq_lih, n_electrons=4, steps=40)
# e_exact_lih = exact_ground_state(H_lih)
# print(f"VQE  : {e_vqe_lih:.6f} Ha")
# print(f"exact: {e_exact_lih:.6f} Ha  (error {abs(e_vqe_lih-e_exact_lih):.2e})")